# `run_secundario` — rotina diária

O que você roda **todo dia**, depois de já ter uma base carregada (`setup_inicial`).

**Janela:** liquidações **D-3 .. D-1** em dias úteis (`D-1` = último pregão fechado). Reprocessa
3 pregões, não 1, porque a B3 lança boletas D+1 com atraso — o que entrou hoje pode alterar
um dia anterior. Tudo é UPSERT, então reprocessar é barato e não duplica.

**Um bloco por fluxo**, na ordem obrigatória (scraping → cálculo → relatório). Nada aborta:
cada passo imprime `[OK]`/`[FALHA]`. A conferência final mostra o que a base ganhou.

⚠️ **Rode a partir da pasta `code/`.** Pode dar `Run All`.

> Equivalente agendável no Task Scheduler: `python scripts/run_diario.py --last 3`.

## Config — rodar primeiro
Calcula a janela sozinho. Só mexa se quiser reprocessar um período diferente.

In [ ]:
import sqlite3
import sys
import time
from datetime import date
from pathlib import Path

if not (Path.cwd() / "scripts").exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "scripts"))
import pipeline_core as pc
from lib.db import ObterBanco

# ==================== EDITE AQUI (so se precisar) ====================
N_PREGOES = 3     # 3 = janela D-3..D-1. Aumente p/ reprocessar mais dias pra tras.
WORKERS   = 8     # threads das chamadas de API (calc_taxa / ntnb)
# =====================================================================

ObterBanco().close()
DB = "data/trades.db"

# D1 = ultimo pregao FECHADO (hoje ainda nao tem boletim/indicativa publicados).
D1   = pc.DiaUtilAnterior(date.today())
DIAS = [d.isoformat() for d in pc.UltimosNDiasUteis(N_PREGOES, ref=D1)]   # [D-3, D-2, D-1]

# XANT = pregao anterior ao 1o dia da janela (D-4).
# Necessario nas fontes casadas por dtNegocio: um trade que LIQUIDA em D-3 pode ter
# sido NEGOCIADO em D-4 (D+1). Sem o boletim/MtM/indicativa de D-4, esse trade some
# ou fica sem spread. Por isso as fontes puxam D-4, mas o CALCULO so roda D-3..D-1.
XANT     = pc.DiaUtilAnterior(DIAS[0]).isoformat()
DIAS_SRC = [XANT] + DIAS

print(f"Base: {Path(DB).resolve()}")
print(f"Hoje = {date.today()}")
print(f"Liquidacoes a processar (D-{N_PREGOES}..D-1): {DIAS[0]} .. {DIAS[-1]}  -> {DIAS}")
print(f"Fontes puxadas (inclui X-1u = {XANT}):        {DIAS_SRC[0]} .. {DIAS_SRC[-1]}")

conferencias = []

def Cobertura(desc, tabela, colunaData, dias, extraSql=""):
    """Quantos dos `dias` esperados ficaram com dados na tabela?"""
    ini, fim = dias[0], dias[-1]
    sql = (f"SELECT COUNT(DISTINCT {colunaData}) FROM {tabela} "
           f"WHERE {colunaData} BETWEEN ? AND ? {extraSql}")
    try:
        n = sqlite3.connect(DB).execute(sql, (ini, fim)).fetchone()[0]
    except Exception as e:
        print(f"[ERRO ] {desc}: {e}")
        conferencias.append((desc, False))
        return False
    ok = n >= len(dias)
    marca = "[OK]    " if ok else ("[PARCIAL]" if n else "[VAZIO] ")
    print(f"{marca} {desc}: {n}/{len(dias)} pregoes ({ini} .. {fim})")
    conferencias.append((desc, ok))
    return ok

## Scraping — 1 bloco por fonte
Janela `DIAS_SRC` (= D-4 .. D-1) nas fontes casadas por `dtNegocio`.

In [ ]:
# 1. Boletim B3 (negocios) -> NegociosBrutos
pc.Boletim(DIAS_SRC[0], DIAS_SRC[-1])
Cobertura("boletim -> NegociosBrutos", "NegociosBrutos", "dtNegocio", DIAS_SRC,
          "AND cdSituacao != 'Cancelado'")

In [ ]:
# 2. Anbima debentures (taxa indicativa) -> AnbimaIndicativos
pc.AnbimaDeb(DIAS_SRC[0], DIAS_SRC[-1])
Cobertura("anbima_deb -> AnbimaIndicativos", "AnbimaIndicativos", "dtReferencia", DIAS_SRC)

In [ ]:
# 3. Anbima CRI/CRA (taxa indicativa, Playwright) -> AnbimaIndicativos
#    O portal so guarda ~5 pregoes — a janela D-4..D-1 cabe.
for d in DIAS_SRC:
    pc.AnbimaCriCra(d)
Cobertura("anbima_cricra -> AnbimaIndicativos (CRI/CRA)", "AnbimaIndicativos", "dtReferencia", DIAS_SRC,
          "AND (cdTicker LIKE 'CRA%' OR cdTicker GLOB '[0-9]*')")

In [ ]:
# 4. FI Analytics planilha (caracteristicas) -> InfoAtivos. Snapshot atual, sem data.
pc.FiAnalytics()
n = sqlite3.connect(DB).execute(
    "SELECT COUNT(*) FROM InfoAtivos WHERE DATE(dtAtualizacao) = DATE('now','localtime')").fetchone()[0]
conferencias.append(("fianalytics -> InfoAtivos (gravado hoje)", n > 0))
print(f"{'[OK]    ' if n else '[VAZIO] '} fianalytics -> InfoAtivos: {n:,} ativo(s) atualizados hoje")

In [ ]:
# 5. Anbima Data (caracteristicas + agenda, Playwright) -> InfoAtivos + FluxoAtivos
#    Incremental: so tickers negociados na janela que ainda tem campo faltando.
pc.AnbimaData(DIAS_SRC[0], DIAS_SRC[-1])
n = sqlite3.connect(DB).execute("SELECT COUNT(*) FROM FluxoAtivos").fetchone()[0]
conferencias.append(("anbima_data -> FluxoAtivos", n > 0))
print(f"{'[OK]    ' if n else '[VAZIO] '} anbima_data -> FluxoAtivos: {n:,} linha(s) na base")

In [ ]:
# 6. Anbima NTN-B (MtM) -> MtmAnbima. Duration em paralelo + skip do ja calculado.
pc.Ntnb(DIAS_SRC[0], DIAS_SRC[-1], workers=WORKERS)
Cobertura("ntnb -> MtmAnbima (NTN-B)", "MtmAnbima", "dtReferencia", DIAS_SRC,
          "AND cdTicker LIKE 'NTN-B%'")

In [ ]:
# 7. Curva DI B3 (MtM) -> MtmAnbima. So aceita data unica -> 1 chamada por pregao.
for d in DIAS_SRC:
    pc.CurvaDi(d)
Cobertura("curva_di -> MtmAnbima (DI1)", "MtmAnbima", "dtReferencia", DIAS_SRC,
          "AND cdTicker LIKE 'DI1%'")

In [ ]:
# 8. Outstanding via Bloomberg -> Outstanding.  *** SO RODA NO BANCO ***
#    No PC pessoal da [FALHA]/[VAZIO] (sem terminal Bloomberg) — esperado.
pc.Outstanding(DIAS_SRC[0], DIAS_SRC[-1])
Cobertura("outstanding -> Outstanding (so no banco)", "Outstanding", "dtOutstanding", DIAS_SRC)

## Cálculo — 1 bloco por fluxo
Só nas liquidações `DIAS` (D-3..D-1). Ordem obrigatória:
taxa → filtrar → spread Anbima → match → spread over → relatório.

In [ ]:
# 9. Taxa por trade (cascata FI Analytics -> B3) -> NegociosProcessados. O passo mais LENTO.
#    Sem --force: trades ja calculados sao pulados (so o que mudou bate em API).
for X in DIAS:
    pc.CalcTaxa(X, workers=WORKERS)
Cobertura("calc_taxa -> vrTaxaCalculada", "NegociosProcessados", "dtLiquidacao", DIAS,
          "AND vrTaxaCalculada IS NOT NULL")

In [ ]:
# 10. Filtrar (VALIDO / FUNDO / BROKER / PF) -> NegociosProcessados.cdStatus
for X in DIAS:
    pc.Filtrar(X)
Cobertura("filtrar -> cdStatus = VALIDO", "NegociosProcessados", "dtLiquidacao", DIAS,
          "AND cdStatus = 'VALIDO'")

In [ ]:
# 11. Spread Anbima das indicativas -> AnbimaIndicativos.vrSpreadAnbima
#     Roda em DIAS_SRC: o relatorio casa a indicativa pela dtNegocio do trade (inclui D-4).
for d in DIAS_SRC:
    pc.SpreadAnbima(d)
Cobertura("spread_anbima -> vrSpreadAnbima", "AnbimaIndicativos", "dtReferencia", DIAS_SRC,
          "AND vrSpreadAnbima IS NOT NULL")

In [ ]:
# 12. Match de referencia (global, sem data) -> InfoAtivos.cdReferencia. ANTES do spread_over.
pc.MatchRef()
n = sqlite3.connect(DB).execute(
    "SELECT COUNT(*) FROM InfoAtivos WHERE cdReferencia IS NOT NULL").fetchone()[0]
conferencias.append(("match_ref -> cdReferencia", n > 0))
print(f"{'[OK]    ' if n else '[VAZIO] '} match_ref -> cdReferencia: {n:,} ativo(s) com referencia")

In [ ]:
# 13. Spread over dos trades (casado por dtNegocio) -> NegociosProcessados.vrSpreadOver
for X in DIAS:
    pc.SpreadOver(X)
Cobertura("spread_over -> vrSpreadOver", "NegociosProcessados", "dtLiquidacao", DIAS,
          "AND vrSpreadOver IS NOT NULL")

In [ ]:
# 14. Gerar relatorio HTML (le a base inteira)
alvo  = Path("data/relatorios/relatorio_secundario.html")
antes = alvo.stat().st_mtime if alvo.exists() else 0
pc.Relatorio()
ok = alvo.exists() and alvo.stat().st_mtime > antes
conferencias.append(("relatorio -> HTML regravado", ok))
if alvo.exists():
    print(f"{'[OK]    ' if ok else '[VAZIO] '} relatorio: {alvo.resolve()} "
          f"({alvo.stat().st_size / 1e6:.1f} MB, gravado ha {time.time() - alvo.stat().st_mtime:.0f}s)")
else:
    print("[VAZIO] relatorio: arquivo nao foi criado")

## Conferência do dia

In [ ]:
oks = sum(1 for _, ok in conferencias if ok)
print(f"{'#' * 62}\n# RODADA {DIAS[0]} .. {DIAS[-1]}: {oks}/{len(conferencias)} conferencias plenas\n{'#' * 62}")
for desc, ok in conferencias:
    print(f"  {'OK   ' if ok else 'FALTA'}  {desc}")

print("\nVolume por liquidacao processada (status VALIDO):")
c = sqlite3.connect(DB)
for X in DIAS:
    row = c.execute("SELECT COUNT(DISTINCT cdTicker), COUNT(*), COALESCE(SUM(vrVolume),0) "
                    "FROM NegociosProcessados WHERE dtLiquidacao = ? AND cdStatus = 'VALIDO'",
                    (X,)).fetchone()
    print(f"  {X}: {row[0]:>5} ativos | {row[1]:>7,} negocios | R$ {row[2] / 1e6:>10,.2f} MM")
c.close()

if oks < len(conferencias):
    print("\nAlgum FALTA? Re-rode so o bloco correspondente (tudo idempotente).")
    print("'outstanding' fica VAZIO fora do banco (sem Bloomberg) — normal.")